In [4]:
import polars as pl
from pathlib import Path

# Check your session's base data
base_data_path = Path(r"C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260319_031558_01\base_data.parquet")

if base_data_path.exists():
    df_base = pl.scan_parquet(base_data_path).select(pl.all().head(1)).collect()
    print("--- BASE DATA COLUMNS ---")
    print(df_base.columns)
    
    # Look for 'bbw' or 'bollinger' names
    bbw_cols = [c for c in df_base.columns if "bbw" in c.lower()]
    print(f"Found BBW columns in source: {bbw_cols}")

In [10]:
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path

# --- CONFIG ---
FILE_PATH = r"C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260319_085558_01\master_metrics.parquet"
FILE_PATH = r"C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260319_085558_01\equity_partitioned\_tmp\equity_era_int=20230901_batch=0_worker=0_pid=8401_1773910663000_da3ae6f2da71.parquet"

def inspect_parquet_sample(path_str: str, n_rows: int = 10):
    path = Path(path_str)
    if not path.exists():
        print(f"❌ File not found: {path}")
        return

    # 1. Open the Parquet file (Metadata only, no data loaded yet)
    parquet_file = pq.ParquetFile(path)
    
    print("="*60)
    print(f"FILE: {path.name}")
    print(f"TOTAL ROWS: {parquet_file.metadata.num_rows:,}")
    print(f"ROW GROUPS: {parquet_file.metadata.num_row_groups}")
    print("="*60)

    # 2. Inspect Schema (Check Data Types)
    print("\n--- Column Schema ---")
    schema = parquet_file.schema_arrow
    for field in schema:
        print(f"{field.name:25} | {field.type}")

    # 3. Read a Sample (Only reads the first few rows/row-group)
    # Using a batch reader to be memory efficient
    print(f"\n--- Sample Data (First {n_rows} rows) ---")
    table_sample = parquet_file.read_row_group(0).to_pandas().head(n_rows)
    print(table_sample.to_string(index=False))

    # 4. Statistical Quick-Check (Check for the "Empty" problem)
    # We read only the specific columns needed to save memory
    stats_cols = ["balance", "max_drawdown", "total_pos"]
    available_cols = [c for c in stats_cols if c in schema.names]
    
    if available_cols:
        subset = parquet_file.read(columns=available_cols).to_pandas()
        print("\n--- Distribution Check ---")
        print(subset.describe().loc[['min', 'max', 'mean']])

if __name__ == "__main__":
    inspect_parquet_sample(FILE_PATH)

FILE: equity_era_int=20230901_batch=0_worker=0_pid=8401_1773910663000_da3ae6f2da71.parquet
TOTAL ROWS: 56,169
ROW GROUPS: 1

--- Column Schema ---
regime_id                 | int32
era_int                   | int64
side                      | int8
ma_int                    | int32
ma_reversion              | bool
entry_lookback_units      | int32
exit_window_h             | int32
use_stochastic            | bool
stoch_key                 | large_string
bbw_periods               | int32
bbw_std                   | float
SL                        | float
TP                        | float
total_pos                 | int32
win_pos                   | int32
balance                   | float
max_drawdown              | float

--- Sample Data (First 10 rows) ---
 regime_id  era_int  side  ma_int ma_reversion  entry_lookback_units  exit_window_h use_stochastic stoch_key  bbw_periods  bbw_std  SL  TP  total_pos  win_pos  balance  max_drawdown
         6 20230901     1     NaN         None      

In [6]:
import polars as pl

df = pl.read_parquet(r"C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260319_085558_01\master_metrics.parquet")

# Count how many rows actually have trade data
valid_rows = df.filter(pl.col("total_pos").is_not_null() & pl.col("total_pos").is_not_nan()).height

print(f"Total Rows: {len(df)}")
print(f"Rows with actual results: {valid_rows}")

if valid_rows > 0:
    print("Sample of valid data:")
    print(df.filter(pl.col("total_pos") > 0).head(5))

InvalidOperationError: `is_not_nan` operation not supported for dtype `i32`

In [ ]:
import polars as pl

# Load the FINAL merged file
df = pl.read_parquet(r"C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260319_085558_01\master_metrics.parquet")

# Check if ANY trades exist in the whole master file
valid_data = df.filter(pl.col("total_pos") > 0)

print(f"Master File Total Rows: {df.height:,}")
print(f"Master File Rows with trades: {valid_data.height:,}")

if valid_data.height > 0:
    print("\n✅ Merge is fine! The data exists.")
    print("The 'NaNs' you saw earlier were just the first few (empty) regimes.")
    print(valid_data.sort("balance", descending=True).head(5))
else:
    print("\n❌ Merge FAILED. The valid data from Batch 17 is missing from the Master file.")

Master File Total Rows: 385,018
Master File Rows with trades: 0

❌ Merge FAILED. The valid data from Batch 17 is missing from the Master file.
